In [112]:
# Settings
MODE = "tfidf" # or "bow"
SIMILARITY = "cosine" # or "euclidean"
K = 8
DATA_SOURCE = "../synthetic_data/templated/templated_pairs_200.json" # "../synthetic_data/API/synthetic_pairs_200.json" 

In [113]:
with open("bag_of_words.txt") as f:
    vocabulary = f.readlines()

In [114]:
vocabulary = [v.strip().lower() for v in vocabulary]
vocabulary

['risk',
 'risky',
 'risks',
 'uncertain',
 'uncertainty',
 'variable',
 'uncertainties',
 'possibility',
 'pending',
 'uncertain',
 'chance',
 'doubt',
 'prospect',
 'exposed',
 'variability',
 'likelihood',
 'threat',
 'probability',
 'bet',
 'varying',
 'unknown',
 'unclear',
 'doubtful',
 'unpredictable',
 'speculative',
 'fear',
 'hesitant',
 'reservation',
 'risky',
 'sticky',
 'instability',
 'tricky',
 'dangerous',
 'tentative',
 'fluctuating',
 'gamble',
 'hazardous',
 'hazard',
 'queries',
 'danger',
 'unstable',
 'erratic',
 'vague',
 'unpredictability',
 'query',
 'unsettled',
 'jeopardize',
 'riskier',
 'irregular',
 'dilemma',
 'hesitancy',
 'unsure',
 'skepticism',
 'unresolved',
 'jeopardy',
 'risking',
 'suspicion',
 'hesitating',
 'halting',
 'peril',
 'risked',
 'unreliable',
 'insecurity',
 'undetermined',
 'apprehension',
 'undecided',
 'wager',
 'precarious',
 'torn',
 'unsafe',
 'unforeseeable',
 'debatable',
 'wavering',
 'riskiest',
 'dicey',
 'endanger',
 'fal

In [115]:
import json
with open(DATA_SOURCE) as f:
    examples = json.load(f)

In [116]:
no_uncertainty = [e["no"] for e in examples]
yes_uncertainty = [e["high"] for e in examples]

In [117]:
if "templated" in DATA_SOURCE:
    train_data = no_uncertainty[1:101] + yes_uncertainty[1:101]
    test_data = no_uncertainty[101:] + yes_uncertainty[101:]
    train_labels = (["low"] * 100) + (["high"] * 100)
    test_labels = (["low"] * 99) + (["high"] * 99)
else:
    train_data = no_uncertainty[:100] + yes_uncertainty[:100]
    test_data = no_uncertainty[100:] + yes_uncertainty[100:]
    train_labels = (["low"] * 100) + (["high"] * 100)
    test_labels = (["low"] * 100) + (["high"] * 100)

In [118]:
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

True

In [119]:
from nltk.tokenize import word_tokenize
def nltk_tokenizer(doc: str) -> list[str]:
    """Tokenize with NLTK and lowercase."""
    return [t.lower() for t in word_tokenize(doc)]

In [120]:
vocab_sorted = sorted(set(w.lower() for w in vocabulary))

In [121]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
if MODE == "tfidf":
    vectorizer = TfidfVectorizer(
        tokenizer=nltk_tokenizer,
        vocabulary=vocab_sorted,
        token_pattern=None,      # suppress default since we use a custom tokenizer
        smooth_idf=True,
        norm="l2",
    )
else:
    vectorizer = CountVectorizer(
        tokenizer=nltk_tokenizer,
        vocabulary=vocab_sorted,
        token_pattern=None,
    )

In [122]:
X_train = vectorizer.fit_transform(train_data)
X_test = vectorizer.transform(test_data)

feature_names = vectorizer.get_feature_names_out()

In [123]:
import pandas as pd
if MODE == "tfidf":
    print(f"\nIDF values (from {len(train_data)} training docs):")
    idf_df = pd.DataFrame(
        [vectorizer.idf_], columns=feature_names, index=["IDF"]
    ).round(4)
    print(idf_df.to_string())



IDF values (from 200 training docs):
     ambivalence  ambivalent  apprehension     bet  chance  chancy  changeability  changeable  danger  dangerous  debatable  defenseless   dicey  diffidence  diffident  dilemma  disquiet   dodgy   doubt  doubtful  dubious  endanger  equivocating  equivocation  erratic  exposed  faltering    fear  fickleness  fitful  fluctuant  fluctuating  gamble  gnarly   hairy  halting  hazard  hazardous    hazy  hesitancy  hesitant  hesitating    iffy  imperil  incalculable  incertitude  indecision  indecisive  insecure  insecurity  instability  irregular  jeopardize  jeopardy  likelihood  menace  misgiving  niggle  oscillating  parlous  pending   peril  perilous  possibility  precarious  probability  prospect   qualm  quandary  queries   query  reservation    risk  risked  riskier  riskiest  riskiness  risking   risks   risky  skepticism  speculative  sticky  suspicion  tentative  tentativeness  threat    torn  treacherous  tricky  uncertain  uncertainties  unc

In [124]:
train_df = pd.DataFrame(
    X_train.toarray(),
    columns=feature_names,
    index=[f"[{train_labels[i]}] Doc {i+1}" for i in range(len(train_labels))],
).round(4)
train_df.head()

,ambivalence,ambivalent,apprehension,bet,chance,chancy,changeability,changeable,danger,dangerous,...,vacillating,vacillation,vague,vagueness,variability,variable,varying,wager,wariness,wavering
[low] Doc 1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
[low] Doc 2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
[low] Doc 3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
[low] Doc 4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
[low] Doc 5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [125]:
test_df = pd.DataFrame(
    X_test.toarray(),
    columns=feature_names,
    index=[f"[{test_labels[i]}] Doc {i+1}" for i in range(len(test_labels))],
).round(4)
test_df.head()

,ambivalence,ambivalent,apprehension,bet,chance,chancy,changeability,changeable,danger,dangerous,...,vacillating,vacillation,vague,vagueness,variability,variable,varying,wager,wariness,wavering
[low] Doc 1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
[low] Doc 2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
[low] Doc 3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
[low] Doc 4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
[low] Doc 5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [126]:
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
if SIMILARITY == "cosine":
    sim_matrix = cosine_similarity(X_test, X_train)
else:
    dist_matrix = euclidean_distances(X_test, X_train)
    sim_matrix = -dist_matrix  # negate so higher = more similar

sim_df = pd.DataFrame(
    sim_matrix.round(4),
    columns=[f"Train {j+1} [{train_labels[j]}]" for j in range(len(train_labels))],
    index=[f"Test {i+1} [{test_labels[i]}] " for i in range(len(test_labels))],
)
sim_df.head()

,Train 1 [low],Train 2 [low],Train 3 [low],Train 4 [low],Train 5 [low],Train 6 [low],Train 7 [low],Train 8 [low],Train 9 [low],Train 10 [low],...,Train 191 [high],Train 192 [high],Train 193 [high],Train 194 [high],Train 195 [high],Train 196 [high],Train 197 [high],Train 198 [high],Train 199 [high],Train 200 [high]
Test 1 [low],0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Test 2 [low],0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Test 3 [low],0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Test 4 [low],0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Test 5 [low],0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [127]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
knn = KNeighborsClassifier(n_neighbors=K, metric=SIMILARITY)
mlp = MLPClassifier()
knn.fit(X_train, train_labels)
mlp.fit(X_train, train_labels)

c:\Users\Claire Schlesinger\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(100,)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",200
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",None


In [128]:
predictions_knn = knn.predict(X_test)
predictions_mlp = mlp.predict(X_test)

In [129]:
from sklearn.metrics import classification_report, confusion_matrix
labels = sorted(set(train_labels))
cm = confusion_matrix(test_labels, predictions_knn, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"True {l}" for l in labels], columns=[f"Pred {l}" for l in labels])
print(cm_df)
print(classification_report(test_labels, predictions_knn, zero_division=0))

           Pred high  Pred low
True high         37        62
True low           0        99
              precision    recall  f1-score   support

        high       1.00      0.37      0.54        99
         low       0.61      1.00      0.76        99

    accuracy                           0.69       198
   macro avg       0.81      0.69      0.65       198
weighted avg       0.81      0.69      0.65       198



In [130]:
labels = sorted(set(train_labels))
cm = confusion_matrix(test_labels, predictions_mlp, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"True {l}" for l in labels], columns=[f"Pred {l}" for l in labels])
print(cm_df)
print(classification_report(test_labels, predictions_mlp, zero_division=0))

           Pred high  Pred low
True high         37        62
True low           0        99
              precision    recall  f1-score   support

        high       1.00      0.37      0.54        99
         low       0.61      1.00      0.76        99

    accuracy                           0.69       198
   macro avg       0.81      0.69      0.65       198
weighted avg       0.81      0.69      0.65       198

